# `CumulativeSplineTrajectory`: Cumulative Lie-group splines

Suppose a robot pose is known at a handful of times, but a sensor asks for the pose *between* those times. We want a curve that is smooth, respects rotations and poses, and can participate in a GTSAM factor graph.

This notebook assumes no prior spline theory. We will start with a smooth scalar switch, use it to blend pose changes, and only then connect the idea to the C++ classes. For a runnable visual example, see [CumulativeSplineTrajectoryExample](../../../python/gtsam/examples/CumulativeSplineTrajectoryExample.ipynb).

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/basis/doc/CumulativeSplineTrajectory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass  # Not in Colab

In [ ]:
from math import comb, factorial

import numpy as np
import plotly.graph_objects as go

## 1. The problem: move smoothly between sparse states

Joining control poses with straight segments gives abrupt changes in velocity at every join. An ordinary weighted average is also problematic: adding rotation matrices or poses does not generally produce a valid rotation or pose.

A cumulative spline takes a different view. Instead of averaging the poses themselves, it records each **change from one control pose to the next** and turns those changes on gradually. The control poses guide the curve; a B-spline generally approximates them rather than passing through every one.

## 2. Start with one smooth switch

Imagine a switch whose value begins at 0, rises smoothly, and ends at 1. Multiplying a change by that switch activates the change without a sudden jump. GTSAM's cubic default, `IrwinHallCDF2`, is such a switch.

The name comes from probability: adding three independent numbers uniformly sampled between 0 and 1 gives an Irwin–Hall distribution. Its cumulative distribution function (CDF) is the smooth blue ramp below; its derivative is the compact orange bump. Here $(x)_+=\max(x,0)$:

$$F_3(t)=\frac{1}{3!}\sum_{k=0}^{3}(-1)^k {3 \choose k}(t-k)_+^3.$$

You do not need the probability interpretation to use the spline. The important facts are that the ramp is 0 before its support, 1 afterward, and has analytic derivatives in between.

In [ ]:
def irwin_hall_cdf(order, time):
    """CDF of the sum of `order` unit-uniform variables."""
    time = np.asarray(time, dtype=float)
    result = np.zeros_like(time)
    for k in range(order + 1):
        result += (-1) ** k * comb(order, k) * np.maximum(time - k, 0.0) ** order
    return np.clip(result / factorial(order), 0.0, 1.0)

time = np.linspace(-0.5, 3.5, 401)
cdf = irwin_hall_cdf(3, time)
pdf = np.gradient(cdf, time)

figure = go.Figure()
figure.add_scatter(x=time, y=cdf, name="smooth activation (CDF)")
figure.add_scatter(x=time, y=pdf, name="activation rate (PDF)")
figure.update_layout(
    title="One change turns on smoothly over a finite interval",
    xaxis_title="kernel coordinate",
    yaxis_title="value",
    template="plotly_white",
)
figure.show()

## 3. Shift one switch for every control-point change

For a scalar sequence, let $\Delta_i=x_i-x_{i-1}$. A cumulative curve is

$$x(t)=x_0+\sum_{i=1}^{N-1}c_i(t)\Delta_i,$$

where $c_i(t)$ is a shifted copy of the smooth ramp. Early in time all switches are off, so the value is $x_0$. As time advances, neighboring switches overlap and several changes contribute at once. This overlap is what removes the sharp corners.

The point density converts physical time into this kernel coordinate. A density of 20 means there are 20 control points per unit of time; time derivatives are rescaled by the corresponding power of 20.

## 4. Replace subtraction and addition with `Log` and `Exp`

Poses and rotations live on a **Lie group**: they can be composed and inverted, but they are not ordinary vectors. Two operations let us temporarily work in a nearby vector space:

| Operation | Plain-language meaning |
|---|---|
| $\operatorname{Log}(T_{i-1}^{-1}T_i)$ | Describe the motion from one pose to the next as a tangent vector. |
| $\operatorname{Exp}(\xi)$ | Turn a tangent vector back into a valid group element. |

For control points $T_0,\ldots,T_{N-1}$, define $\xi_i=\operatorname{Log}(T_{i-1}^{-1}T_i)$. The implementation builds

$$T(t)=T_0\operatorname{Exp}\left(\sum_{i=1}^{N-1}c_i(t)\xi_i\right).$$

This is the same cumulative idea as the scalar equation: compute consecutive changes, smoothly weight them, add them in tangent coordinates, and map the result back to a valid pose. Analytically differentiating the switches yields smooth tangent-rate and higher-derivative expressions without finite differences.

## 5. How the GTSAM pieces correspond to that story

| Idea above | GTSAM component |
|---|---|
| A smooth switch and its derivatives | `KernelBase` defines the small kernel interface. |
| Exact formulas on successive intervals | `PiecewisePolynomial<Order, Pieces>` stores and differentiates each polynomial segment. |
| Ready-made B-spline switches | `kernels::IrwinHallCDF*` supplies cumulative kernels; `IrwinHallCDF2` is the cubic default. |
| Ordinary scalar or vector basis weights | `CardinalSplineBasis` connects the cubic kernel to existing `Basis` functors and factors. |
| Poses, rotations, or other Lie-group values | `CumulativeSplineTrajectory<T>` creates expression-valued samples and tangent derivatives. |

Use `CardinalSplineBasis` when the sample coordinate is known and the unknowns are ordinary scalar or vector coefficients. Use `CumulativeSplineTrajectory` when the values lie on a Lie group, or when the timestamp itself must remain a differentiable expression.

## 6. C++ trajectory usage

This example creates a pose expression at a time that is itself an optimization variable. That is useful, for example, when estimating a sensor clock offset together with a trajectory.

```cpp
CumulativeSplineTrajectory<Pose3> trajectory(20.0);
for (size_t i = 0; i < poseCount; ++i) {
  trajectory.addControlPoint(Pose3_(Symbol('p', i)));
}

Double_ time(Symbol('t', 0));
Pose3_ pose = trajectory.sampleTrajectory(time, 4.5, 5.5);
Vector6_ tangentRate =
    trajectory.sampleTrajectoryDerivative(time, 4.5, 5.5, 1);
```

The optional window says that `time` is expected between 4.5 and 5.5. It keeps unrelated control points out of the expression graph, which is important for sparsity. The kernel is held by reference and must outlive the trajectory; the exported Irwin–Hall kernels have static lifetime.

## 7. Fixed-coordinate basis usage

When the coordinate is already known, the same cubic construction can be represented by ordinary dense weights and reused by the existing basis factors:

```cpp
Vector controlPoints = (Vector(4) << 0.0, 1.0, 0.0, 1.0).finished();
CardinalSplineBasis::EvaluationFunctor evaluate(4, 2.5);
double value = evaluate(controlPoints);

auto noise = noiseModel::Isotropic::Sigma(1, 0.1);
EvaluationFactor<CardinalSplineBasis> factor(
    Symbol('c', 0), measurement, noise, 4, 2.5);
```

The basis form is linear in scalar or vector control values. The trajectory form is nonlinear and works directly with Lie-group control points.

## 8. Where to go next

- Run the [Pose2 example notebook](../../../python/gtsam/examples/CumulativeSplineTrajectoryExample.ipynb) to see cubic basis weights, a smooth planar path, and tangent-rate plots computed by the generated `CumulativeSplineTrajectoryPose2` and `CardinalSplineBasis` Python wrappers.
- Read [AsVectorSpace](../../geometry/doc/AsVectorSpace.ipynb) if a component such as camera calibration is a manifold but not naturally a Lie group.
- `CartesianProduct<A, B>` combines compatible components, for example a pose and an explicitly adapted calibration.

## Source

- [CumulativeSplineTrajectory.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CumulativeSplineTrajectory.h)
- [CardinalSplineBasis.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CardinalSplineBasis.h)
- [PiecewisePolynomial.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/PiecewisePolynomial.h)
- [IrwinHall.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/IrwinHall.h)